# AP01 – JOINS

In [2]:
# Libraries
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

# Current working directory
print('Arbeitsverzeichnis:', Path.cwd())

#NICHTS ANPASSEN: Imports und Path.cwd()
#ERKENNTNIS: Relative Dateipfade beziehen sich auf das Arbeitsverzeichnis des Notebooks

Arbeitsverzeichnis: /workspaces/DS_HS2026_LNW/DS_HS2026_LNW_I_Examples/AP01/AP01 Mock Exam


In [3]:
# Set up database connection
user = 'pgadmin'                                                                #Benutzername für PostgreSQL
password = 'geheim'                                                             #Passwort des Datenbankbenutzers
host = 'localhost'                                                              #Adresse des PostgreSQL-Servers
port = '5432'                                                                   #Port, über den PostgreSQL erreichbar ist
database = 'postgres'                                                           #Name der Datenbank, mit der verbunden wird

# Create Connection URL
db_connection_url = (
    f'postgresql://{user}:{password}@{host}:{port}/{database}'                  #Verbindungsadresse aus den oben definierten Angaben zusammensetzen
)                                                                               

# Create SQLAlchemy Engine
engine = create_engine(db_connection_url)                                       #Engine erstellen; Verbindung wird erst bei Bedarf geöffnet

# Test database connection
with engine.connect() as connection:                                            #Verbindung öffnen und nach dem Block automatisch schliessen
    result = connection.execute(text('SELECT current_database()'))              #Namen der aktuellen Datenbank abfragen
    print('Verbundene Datenbank:', result.scalar())                             #Einzelnen Ergebniswert auslesen und anzeigen

# Dispose the engine
engine.dispose()                                                                #Verbindungen im Pool schliessen und Ressourcen freigeben

Verbundene Datenbank: postgres


## Rules

### Grundprinzip
- JOINs verbinden Zeilen mehrerer Tabellen über zusammengehörende Spalten
- Meist wird ein Fremdschlüssel mit dem passenden Primärschlüssel verbunden
- `ON` bestimmt die Verknüpfung, `SELECT` bestimmt die ausgegebenen Spalten
- Die JOIN-Spalten müssen nicht im SELECT stehen
- Gleiche Spaltennamen allein sind kein Beweis für eine passende Beziehung. Schema und Bedeutung prüfen

### Welcher JOIN für welche Frage?
| JOIN | Welche Zeilen bleiben erhalten? |
|---|---|
| `INNER JOIN` | Nur passende Kombinationen beider Tabellen |
| `LEFT JOIN` | Alle linken Zeilen und passende rechte Zeilen |
| `RIGHT JOIN` | Alle rechten Zeilen und passende linke Zeilen |
| `FULL OUTER JOIN` | Alle Zeilen beider Tabellen |
| `CROSS JOIN` | Jede linke Zeile kombiniert mit jeder rechten Zeile |
- `JOIN` ohne Zusatz bedeutet `INNER JOIN`
- `FULL JOIN` und `FULL OUTER JOIN` bedeuten dasselbe
- Bei OUTER JOINs wird die fehlende Gegenseite mit NULL ergänzt
- Ein `RIGHT JOIN` lässt sich durch Vertauschen der Tabellen als `LEFT JOIN` schreiben
- `CROSS JOIN` benötigt keine ON-Bedingung. Bei n und m Zeilen entstehen n × m Kombinationen

### Tabellen und Spalten eindeutig ansprechen
- Tabellenaliase machen Abfragen kürzer, bspw. `sales AS s` und `salespersons AS sp`
- Nach Vergabe eines Tabellenalias diesen für die Tabelle verwenden
- Gleichnamige Spalten mit dem Tabellenalias kennzeichnen, bspw. `s.salespersonid`
- Für gleichnamige Ausgabespalten unterschiedliche Aliase vergeben, bspw. `customer_firstname` und `salesperson_firstname`

### Alle Objekte behalten
- Formulierungen wie «alle Verkäufer, auch ohne Verkauf» verlangen meist einen `LEFT JOIN`
- Die vollständig benötigte Tabelle muss links stehen, bspw. `FROM salespersons sp LEFT JOIN sales s`
- Bei mehreren JOINs den gesamten Verknüpfungsweg prüfen. Ein nachfolgender INNER JOIN kann zuvor erhaltene Zeilen wieder ausschliessen
- Ein LEFT JOIN liefert nicht zwingend genau eine Zeile pro linkem Objekt. Mehrere rechte Treffer erzeugen mehrere Ergebniszeilen

### Objekte ohne passende Verbindung finden
- «Noch nie», «ohne» oder «keine zugehörigen Einträge» meist `LEFT JOIN ... WHERE rechte_id IS NULL`
- Für die NULL-Prüfung eine rechte Spalte verwenden, die bei echten Treffern garantiert nicht NULL ist, bspw. den Primärschlüssel
- Keine beliebige rechte Spalte wie Preis verwenden. Diese könnte auch bei einem vorhandenen Treffer fehlen
- `INNER JOIN ... WHERE rechte_id IS NULL` findet bei einer nicht nullable rechten ID keine Objekte ohne Treffer, weil diese bereits ausgeschlossen wurden
- Alternative: `NOT EXISTS` prüft, ob kein passender Eintrag vorhanden ist
- Nur unverknüpfte Zeilen beider Seiten: `FULL OUTER JOIN` mit `WHERE linke_id IS NULL OR rechte_id IS NULL`

### ON oder WHERE beim LEFT JOIN?
- `ON` bestimmt, welche rechten Zeilen als Treffer gelten
- `WHERE` filtert das Ergebnis nach dem JOIN
- Sollen alle linken Objekte erhalten bleiben, rechte Zusatzbedingungen meist in `ON` setzen
- Bspw. alle Verkäufer inkl. jener ohne Verkäufe im Jahr 2023, Datumsfilter der Verkäufe in `ON`
- Derselbe Datumsfilter in `WHERE` entfernt die ergänzten NULL-Zeilen und damit Verkäufer ohne passenden Verkauf

### JOINs mit COUNT und SUM
- `COUNT(*)` zählt Ergebniszeilen, auch durch LEFT JOIN ergänzte Zeilen ohne rechten Treffer
- Für die Anzahl echter rechter Treffer `COUNT(rechte_id)` verwenden, ohne Treffer ergibt das 0
- Die gezählte ID muss bei echten Treffern garantiert vorhanden sein
- `SUM`, `AVG`, `MIN` und `MAX` ignorieren NULL und liefern ohne bekannte Werte NULL
- Falls fachlich passend, fehlende Summen durch 0 ersetze, bspw. `COALESCE(SUM(s.saleprice), 0)`
- Nach der ID des ausgewerteten Objekts gruppieren, nicht nur nach Namen. Unterschiedliche Personen können gleich heissen
- Nicht aggregierte SELECT-Spalten grundsätzlich ebenfalls in GROUP BY aufnehmen

### Zeilenvervielfachung und falsche Kennzahlen
- Bei 1:n-Beziehungen erscheint die linke Zeile einmal pro passendem rechten Treffer
- Mehrere unabhängige 1:n-JOINs können Treffer miteinander vervielfachen und Anzahlen oder Summen erhöhen
- Bspw. zwei Verkäufe und drei Services desselben Fahrrads können zusammen sechs Ergebniszeilen erzeugen
- Bei Bedarf zuerst je Tabelle aggregieren und danach die Ergebnisse verbinden
- `COUNT(DISTINCT id)` zählt eindeutige Objekte, wenn genau diese Anzahl gesucht ist
- `SUM(DISTINCT betrag)` ist keine allgemeine Reparatur. Zwei echte Verkäufe können denselben Betrag haben
- `SELECT DISTINCT` ersetzt keine korrekte JOIN-Bedingung und kann echte Geschäftsfälle verdecken

### Mehrere Tabellen verbinden
- Dem Verknüpfungsweg im Schema folgen, bspw. `cinemas → screenings → tickets`
- Jede hinzugefügte Tabelle braucht eine passende Verbindung, ausser beim beabsichtigten CROSS JOIN
- Nicht direkt verwandte IDs miteinander zu vergleichen kann zufällige Treffer und fachlich falsche Ergebnisse erzeugen
- Bei JOIN-Bedingungen mit `=` passen NULL-Schlüssel nicht aufeinander

### Ergebnis prüfen
- Sind alle geforderten Objekte enthalten, auch jene ohne Treffer?
- Werden Objekte ohne Treffer korrekt mit 0 statt 1 gezählt?
- Entspricht die Zeilenanzahl den erwarteten Beziehungen?
- Wurden Summen oder Anzahlen durch zusätzliche JOINs vervielfacht?
- Fehlende Sortierung ergänzen, wenn die Aufgabe eine Reihenfolge verlangt
- Merksatz: Zuerst festlegen, welche Objekte erhalten bleiben müssen, dann den JOIN-Typ und die Schlüssel wählen, anschliessend filtern und aggregieren

## Queries

#### WICHTIG
- Bspw. Format: `SELECT Tabelle1.*, Tabelle2.* FROM Tabelle1 INNER JOIN Tabelle2 ON Tabelle1.GemeinsameSpalte = Tabelle2.GemeinsameSpalte;`
- `Tabelle1.*` und `Tabelle2.*` wählen alle Spalten der jeweiligen Tabelle aus
- Einzelne Spalten gezielt auswählen, bspw. `SELECT base_price, paid_price`
- Kommt ein Spaltenname in beiden Tabellen vor, den Tabellennamen voranstellen, bspw. `screenings.screeningid`!!

### Retrieve each sale's ID, sale price and the first and last name of the salesperson who made the sale
- `INNER JOIN` verbindet Verkäufe mit den passenden Verkäufern über `salespersonid`, nur passende Kombinationen bleiben erhalten
- Format: `SELECT Tabelle1.*, Tabelle2.* FROM Tabelle1 INNER JOIN Tabelle2 ON Tabelle1.GemeinsameSpalte = Tabelle2.GemeinsameSpalte;`

In [4]:
engine = create_engine(db_connection_url)                                       #Engine für den Datenbankzugriff erstellen

df_spickap01 = pd.read_sql_query("""
    SELECT sales.saleid,                                                        -Verkaufs-ID auswählen
           sales.saleprice,                                                     --Verkaufspreis auswählen
           salespersons.firstname,                                              --Vornamen des Verkäufers auswählen
           salespersons.lastname                                                --Nachnamen des Verkäufers auswählen
    FROM sales                                                                  --Verkaufstabelle verwenden
    INNER JOIN salespersons                                                     --Verkäufertabelle ergänzen
        ON sales.salespersonid = salespersons.salespersonid;                    --Über die Verkäufer-ID verbinden und nur passende Kombinationen behalten
""", con=engine)                                                                #Abfrage ausführen und Ergebnis speichern

engine.dispose()                                                                #Verbindungen im Pool freigeben
display(df_spickap01)                                                           #Ergebnistabelle anzeigen

,saleid,saleprice,firstname,lastname
0,SLBF6MT8J2,3480,Robert,Lyons
1,SQX72758TF,2680,Heather,Brown
2,S5FDZ50KNW,3000,Michael,Zhang
3,S79LBQUU41,4320,Lauren,Robinson
4,SIECZIGLOC,1900,Karen,Randall
...,...,...,...,...
679,STJ906IYGY,3120,Michael,Zhang
680,S1L1CI9G7K,1550,Heather,Brown
681,SJQ2VP6CY4,3360,Karen,Randall
682,SJ3XI5MJ1Q,1630,Brian,Klein


### Determine the number of bicycles sold and total sales revenue for each salesperson who has made a sale. Display their first name, last name, phone, email, total_bicycles_sold and total_sales_amount. Round the revenue to two decimal places and sort it in descending order
- `INNER JOIN` verbindet Verkäufer mit ihren Verkäufen, Verkäufer ohne Verkauf werden ausgeschlossen
- `GROUP BY` bildet eine Gruppe pro Verkäufer, `COUNT` zählt Verkäufe und `SUM` addiert Verkaufspreise, der Umsatz wird gerundet und absteigend sortiert
- Format: `SELECT Tabelle1.*, Tabelle2.* FROM Tabelle1 INNER JOIN Tabelle2 ON Tabelle1.GemeinsameSpalte = Tabelle2.GemeinsameSpalte;`

In [19]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT salespersons.firstname,                                                  --Vornamen des Verkäufers anzeigen
           salespersons.lastname,                                                   --Nachnamen des Verkäufers anzeigen
           salespersons.phone,                                                      --Telefonnummer anzeigen
           salespersons.email,                                                      --E-Mail-Adresse anzeigen
           COUNT(sales.saleid) AS total_bicycles_sold,                              --Verkäufe pro Verkäufer zählen
           ROUND(CAST(SUM(sales.saleprice) AS NUMERIC), 2) AS total_sales_amount    --Umsatz pro Verkäufer addieren und auf zwei Stellen runden
    FROM salespersons                                                               --Verkäufertabelle verwenden
    INNER JOIN sales                                                                --Verkäufe ergänzen, nur Verkäufer mit passenden Verkäufen behalten
        ON salespersons.salespersonid = sales.salespersonid                         --Tabellen über die Verkäufer-ID verbinden
    GROUP BY salespersons.salespersonid,
             salespersons.firstname,
             salespersons.lastname,
             salespersons.phone,
             salespersons.email                                                     --Nach Verkäufer-ID und allen nicht aggregierten Ausgabespalten gruppieren
    ORDER BY total_sales_amount DESC;                                               --Verkäufer mit dem höchsten Gesamtumsatz zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,firstname,lastname,phone,email,total_bicycles_sold,total_sales_amount
0,Heather,Brown,+41 (0)46 288 75 19,heather.brown@example.com,82,267840.0
1,Christine,Harris,+41 (0)61 523 86 58,christine.harris@example.com,74,237420.0
2,Karen,Randall,+41 (0)66 616 17 40,karen.randall@example.com,75,232950.0
3,Robert,Lyons,+41 (0)40 422 23 33,robert.lyons@example.com,69,232330.0
4,Lauren,Robinson,+41 (0)71 214 12 13,lauren.robinson@example.com,69,216840.0
5,Ryan,Baxter,+41 (0)40 706 53 70,ryan.baxter@example.com,72,208490.0
6,Nicole,Carter,+41 (0)77 266 76 74,nicole.carter@example.com,66,207940.0
7,Cathy,Becker,+41 (0)23 263 98 44,cathy.becker@example.com,67,200340.0
8,Michael,Zhang,+41 (0)63 311 51 92,michael.zhang@example.com,59,180090.0
9,Brian,Klein,+41 (0)77 449 26 62,brian.klein@example.com,51,174610.0


### Determine the number of sales and total sales revenue for every salesperson, including those with no sales. Display salespersonid, firstname, lastname, sales_count and total_sales_amount. Show zero for salespeople with no sales, round revenue to two decimal places and sort by revenue in descending order
- `LEFT JOIN` behält alle Verkäufer, `COUNT(s.saleid)` zählt nur echte Verkäufe und liefert ohne Verkauf 0
- `COALESCE(SUM(s.saleprice), 0)` ersetzt fehlenden Umsatz durch 0, der Umsatz wird pro Verkäufer berechnet, gerundet und absteigend sortiert
- Format: `SELECT Tabelle1.*, Tabelle2.* FROM Tabelle1 LEFT JOIN Tabelle2 ON Tabelle1.GemeinsameSpalte = Tabelle2.GemeinsameSpalte;`

In [20]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT salespersons.salespersonid,                                          --Verkäufer-ID anzeigen
           salespersons.firstname,                                              --Vornamen anzeigen
           salespersons.lastname,                                               --Nachnamen anzeigen
           COUNT(sales.saleid) AS sales_count,                                  --Nur vorhandene Verkäufe zählen, ohne Verkauf ergibt sich 0
           ROUND(CAST(COALESCE(SUM(sales.saleprice), 0) AS NUMERIC), 2) AS total_sales_amount  --Umsatz summieren, fehlende Summe durch 0 ersetzen und runden
    FROM salespersons                                                           --Von der Tabelle aller Verkäufer ausgehen
    LEFT JOIN sales                                                             --Alle Verkäufer behalten, auch ohne passenden Verkauf
        ON salespersons.salespersonid = sales.salespersonid                     --Verkäufe über die Verkäufer-ID zuordnen
    GROUP BY salespersons.salespersonid,
             salespersons.firstname,
             salespersons.lastname                                              --Eine Gruppe pro Verkäufer bilden
    ORDER BY total_sales_amount DESC;                                           --Höchsten Gesamtumsatz zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,salespersonid,firstname,lastname,sales_count,total_sales_amount
0,SPCZZUOCF9,Heather,Brown,82,267840.0
1,SPGK07EO84,Christine,Harris,74,237420.0
2,SPOHEVXP0T,Karen,Randall,75,232950.0
3,SP0LU6X4G6,Robert,Lyons,69,232330.0
4,SPCGP1HZPG,Lauren,Robinson,69,216840.0
5,SPLO7OLQSK,Ryan,Baxter,72,208490.0
6,SPL5PTPR1D,Nicole,Carter,66,207940.0
7,SPOELVJOXX,Cathy,Becker,67,200340.0
8,SP86XCRI7B,Michael,Zhang,59,180090.0
9,SPXRVM1B8C,Brian,Klein,51,174610.0


### Retrieve the ID, first name and last name of all salespeople who have never made a sale
- `LEFT JOIN` behält alle Verkäufer, `WHERE s.saleid IS NULL` wählt nur jene ohne Verkauf aus
- Format: `SELECT Tabelle1.* FROM Tabelle1 LEFT JOIN Tabelle2 ON Tabelle1.GemeinsameSpalte = Tabelle2.GemeinsameSpalte WHERE Tabelle2.GemeinsameSpalte IS NULL;`

In [21]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT salespersons.salespersonid,                                          --Verkäufer-ID auswählen
           salespersons.firstname,                                              --Vornamen auswählen
           salespersons.lastname                                                --Nachnamen auswählen
    FROM salespersons                                                           --Von der Tabelle aller Verkäufer ausgehen
    LEFT JOIN sales                                                             --Alle Verkäufer behalten, auch ohne passenden Verkauf
        ON salespersons.salespersonid = sales.salespersonid                     --Verkäufe über die Verkäufer-ID zuordnen
    WHERE sales.saleid IS NULL;                                                 --Nur Verkäufer ohne passenden Verkauf behalten
""", con=engine)

engine.dispose()
display(df_spickap01)

,salespersonid,firstname,lastname


### Retrieve every salesperson’s ID, first name and last name together with their sale IDs, sale dates and sale prices from 2023. Include salespeople with no sales in 2023
- `LEFT JOIN` behält alle Verkäufer, der Datumsfilter für 2023 gehört in `ON`, damit Verkäufer ohne passenden Verkauf erhalten bleiben
- Ohne Verkauf in 2023 sind die Verkaufsspalten NULL, derselbe Datumsfilter in `WHERE` würde diese Verkäufer ausschliessen
- Format: `SELECT Tabelle1.*, Tabelle2.* FROM Tabelle1 LEFT JOIN Tabelle2 ON Tabelle1.GemeinsameSpalte = Tabelle2.GemeinsameSpalte;`

In [22]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT salespersons.salespersonid,                                          --Verkäufer-ID anzeigen
           salespersons.firstname,                                              --Vornamen anzeigen
           salespersons.lastname,                                               --Nachnamen anzeigen
           sales.saleid,                                                        --Verkaufs-ID anzeigen, ohne passenden Verkauf NULL
           sales.saledate,                                                      --Verkaufsdatum anzeigen, ohne passenden Verkauf NULL
           sales.saleprice                                                      --Verkaufspreis anzeigen, ohne passenden Verkauf NULL
    FROM salespersons                                                           --Von allen Verkäufern ausgehen
    LEFT JOIN sales                                                             --Alle Verkäufer behalten, auch ohne Verkauf im Jahr 2023
        ON salespersons.salespersonid = sales.salespersonid                     --Verkäufe über die Verkäufer-ID zuordnen
       AND sales.saledate >= '2023-01-01'                                       --Nur Verkäufe ab Jahresbeginn 2023 zuordnen
       AND sales.saledate < '2024-01-01'                                        --Verkäufe ab 2024 ausschliessen, gesamtes Jahr 2023 erfassen
    ORDER BY salespersons.salespersonid, sales.saledate, sales.saleid;          --Nach Verkäufer, Verkaufsdatum und Verkaufs-ID sortieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,salespersonid,firstname,lastname,saleid,saledate,saleprice
0,SP0LU6X4G6,Robert,Lyons,SKVZK3HAN1,2023-01-07,3340
1,SP0LU6X4G6,Robert,Lyons,SZWDVUC0UK,2023-01-12,3980
2,SP0LU6X4G6,Robert,Lyons,S2ECYM73GW,2023-02-15,3200
3,SP0LU6X4G6,Robert,Lyons,S2DAJH0XZ6,2023-02-20,4350
4,SP0LU6X4G6,Robert,Lyons,SZEC9LNCC8,2023-03-29,1780
...,...,...,...,...,...,...
148,SPXRVM1B8C,Brian,Klein,S5UKCEDTVQ,2023-10-08,2580
149,SPXRVM1B8C,Brian,Klein,SR33DQ32G4,2023-10-14,4860
150,SPXRVM1B8C,Brian,Klein,SNU5LWKVBI,2023-10-17,2950
151,SPXRVM1B8C,Brian,Klein,SM9E3EV71W,2023-11-06,2610


### Retrieve every salesperson's ID, first name and last name, together with their sale IDs and sale prices. Include salespeople with no sales
- `RIGHT JOIN` behält alle Verkäufer aus der rechten Tabelle, ohne Verkauf sind Verkaufs-ID und Verkaufspreis NULL
- Format: `SELECT Tabelle1.*, Tabelle2.* FROM Tabelle1 RIGHT JOIN Tabelle2 ON Tabelle1.GemeinsameSpalte = Tabelle2.GemeinsameSpalte;`

In [23]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT salespersons.salespersonid,                                          --Verkäufer-ID auswählen
           salespersons.firstname,                                              --Vornamen auswählen
           salespersons.lastname,                                               --Nachnamen auswählen
           sales.saleid,                                                        --Zugehörige Verkaufs-ID anzeigen
           sales.saleprice                                                      --Zugehörigen Verkaufspreis anzeigen
    FROM sales                                                                  --Verkäufe stehen auf der linken Seite
    RIGHT JOIN salespersons                                                     --Alle Verkäufer der rechten Tabelle behalten
        ON sales.salespersonid = salespersons.salespersonid;                    --Verkäufe zuordnen, ohne Verkauf bleiben die Verkaufsspalten NULL
""", con=engine)

engine.dispose()
display(df_spickap01)

,salespersonid,firstname,lastname,saleid,saleprice
0,SP0LU6X4G6,Robert,Lyons,SLBF6MT8J2,3480
1,SPCZZUOCF9,Heather,Brown,SQX72758TF,2680
2,SP86XCRI7B,Michael,Zhang,S5FDZ50KNW,3000
3,SPCGP1HZPG,Lauren,Robinson,S79LBQUU41,4320
4,SPOHEVXP0T,Karen,Randall,SIECZIGLOC,1900
...,...,...,...,...,...
679,SP86XCRI7B,Michael,Zhang,STJ906IYGY,3120
680,SPCZZUOCF9,Heather,Brown,S1L1CI9G7K,1550
681,SPOHEVXP0T,Karen,Randall,SJQ2VP6CY4,3360
682,SPXRVM1B8C,Brian,Klein,SJ3XI5MJ1Q,1630


### Retrieve all sales and all salespeople, including unmatched records from either table. Display the sale ID, sale price, salesperson ID from each table, first name and last name
- `FULL OUTER JOIN` behält alle Verkäufe und Verkäufer, passende Datensätze werden verbunden und fehlende Gegenseiten mit NULL ergänzt
- Format: `SELECT Tabelle1.*, Tabelle2.* FROM Tabelle1 FULL OUTER JOIN Tabelle2 ON Tabelle1.GemeinsameSpalte = Tabelle2.GemeinsameSpalte;`

In [24]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT sales.saleid,                                                        --Verkaufs-ID auswählen
           sales.saleprice,                                                     --Verkaufspreis auswählen
           sales.salespersonid AS sale_salespersonid,                           --Im Verkauf gespeicherte Verkäufer-ID anzeigen
           salespersons.salespersonid AS salesperson_id,                        --Verkäufer-ID aus der Verkäufertabelle anzeigen
           salespersons.firstname,                                              --Vornamen des Verkäufers auswählen
           salespersons.lastname                                                --Nachnamen des Verkäufers auswählen
    FROM sales                                                                  --Verkaufstabelle auf der linken Seite verwenden
    FULL OUTER JOIN salespersons                                                --Alle Datensätze beider Tabellen behalten
        ON sales.salespersonid = salespersons.salespersonid;                    --Passende Datensätze verbinden, fehlende Gegenseite mit NULL ergänzen
""", con=engine)

engine.dispose()
display(df_spickap01)

,saleid,saleprice,sale_salespersonid,salesperson_id,firstname,lastname
0,SLBF6MT8J2,3480,SP0LU6X4G6,SP0LU6X4G6,Robert,Lyons
1,SQX72758TF,2680,SPCZZUOCF9,SPCZZUOCF9,Heather,Brown
2,S5FDZ50KNW,3000,SP86XCRI7B,SP86XCRI7B,Michael,Zhang
3,S79LBQUU41,4320,SPCGP1HZPG,SPCGP1HZPG,Lauren,Robinson
4,SIECZIGLOC,1900,SPOHEVXP0T,SPOHEVXP0T,Karen,Randall
...,...,...,...,...,...,...
679,STJ906IYGY,3120,SP86XCRI7B,SP86XCRI7B,Michael,Zhang
680,S1L1CI9G7K,1550,SPCZZUOCF9,SPCZZUOCF9,Heather,Brown
681,SJQ2VP6CY4,3360,SPOHEVXP0T,SPOHEVXP0T,Karen,Randall
682,SJ3XI5MJ1Q,1630,SPXRVM1B8C,SPXRVM1B8C,Brian,Klein


### Retrieve sales without a matching salesperson and salespeople without any sales. Display the sale ID, sale price, salesperson ID from each table, first name and last name
- `FULL OUTER JOIN` mit NULL-Prüfung auf beiden Seiten zeigt nur Verkäufe ohne passenden Verkäufer und Verkäufer ohne Verkauf, passende Kombinationen werden ausgeschlossen
- Format: `SELECT Tabelle1.*, Tabelle2.* FROM Tabelle1 FULL OUTER JOIN Tabelle2 ON Tabelle1.GemeinsameSpalte = Tabelle2.GemeinsameSpalte WHERE Tabelle1.GemeinsameSpalte IS NULL OR Tabelle2.GemeinsameSpalte IS NULL;`

In [25]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT sales.saleid,                                                        --Verkaufs-ID auswählen
           sales.saleprice,                                                     --Verkaufspreis auswählen
           sales.salespersonid AS sale_salespersonid,                           --Im Verkauf gespeicherte Verkäufer-ID anzeigen
           salespersons.salespersonid AS salesperson_id,                        --Verkäufer-ID aus der Verkäufertabelle anzeigen
           salespersons.firstname,                                              --Vornamen auswählen
           salespersons.lastname                                                --Nachnamen auswählen
    FROM sales                                                                  --Verkaufstabelle auf der linken Seite verwenden
    FULL OUTER JOIN salespersons                                                --Zunächst alle Datensätze beider Tabellen behalten
        ON sales.salespersonid = salespersons.salespersonid                     --Datensätze über die Verkäufer-ID verbinden
    WHERE sales.saleid IS NULL                                                  --Verkäufer ohne passenden Verkauf auswählen
       OR salespersons.salespersonid IS NULL;                                   --Zusätzlich Verkäufe ohne passenden Verkäufer auswählen
""", con=engine)

engine.dispose()
display(df_spickap01)

,saleid,saleprice,sale_salespersonid,salesperson_id,firstname,lastname


### Retrieve all possible combinations of mechanics and bicycles. Display mechanicid, firstname, lastname, bicycleid, make and model
- `CROSS JOIN` kombiniert jeden Mechaniker mit jedem Fahrrad, bei n Mechanikern und m Fahrrädern entstehen n × m Zeilen ohne ON-Bedingung
- Format: `SELECT Tabelle1.spalte1, Tabelle2.spalte2 FROM Tabelle1 CROSS JOIN Tabelle2;`

In [26]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT mechanics.mechanicid,                                                --Mechaniker-ID auswählen
           mechanics.firstname,                                                 --Vornamen des Mechanikers auswählen
           mechanics.lastname,                                                  --Nachnamen des Mechanikers auswählen
           bicycles.bicycleid,                                                  --Fahrrad-ID auswählen
           bicycles.make,                                                       --Fahrradmarke auswählen
           bicycles.model                                                       --Fahrradmodell auswählen
    FROM mechanics                                                              --Mechanikertabelle verwenden
    CROSS JOIN bicycles;                                                        --Jeden Mechaniker mit jedem Fahrrad kombinieren, keine ON-Bedingung
""", con=engine)

engine.dispose()
display(df_spickap01)

,mechanicid,firstname,lastname,bicycleid,make,model
0,MV26A9TGJL,Erin,Williamson,C7ZGWFLBUV,Trek,Domane
1,MTA6ED59YC,Melissa,Miller,C7ZGWFLBUV,Trek,Domane
2,MBCL24BW42,Bryan,White,C7ZGWFLBUV,Trek,Domane
3,M8DA1H43YN,Shannon,Larsen,C7ZGWFLBUV,Trek,Domane
4,M8XUQMDR0P,Courtney,Ford,C7ZGWFLBUV,Trek,Domane
...,...,...,...,...,...,...
19995,M06GWF4R58,Tiffany,Wilson,CNJYADLN8R,Bianchi,Aria
19996,MLNW721B7L,Bradley,Mckinney,CNJYADLN8R,Bianchi,Aria
19997,MA4MHCDLIX,Cassandra,Sanford,CNJYADLN8R,Bianchi,Aria
19998,MCNI3BOSV1,Lisa,Hall,CNJYADLN8R,Bianchi,Aria


### Retrieve Bianchi bicycles sold from March to December 2023, including their bicycle ID, make, model, sale price, sale date, customer names and salesperson names. Sort by sale date in ascending order and limit the result to 10 rows
- Drei INNER JOINs verbinden Fahrräder mit Verkäufen, Kunden und Verkäufern, `WHERE` filtert auf Bianchi und den Zeitraum März bis Dezember 2023
- `ORDER BY saledate ASC` sortiert chronologisch, `LIMIT 10` begrenzt die Ausgabe auf zehn Zeilen
- Format: `SELECT Tabelle1.*, Tabelle2.* FROM Tabelle1 INNER JOIN Tabelle2 ON Tabelle1.GemeinsameSpalte = Tabelle2.GemeinsameSpalte;`, für weitere Tabellen jeweils einen passenden INNER JOIN ergänzen (bei 5 Tabellen braucht man bspw. 4 Joins)

In [27]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycles.bicycleid,                                                  --Fahrrad-ID auswählen
           bicycles.make,                                                       --Fahrradmarke auswählen
           bicycles.model,                                                      --Fahrradmodell auswählen
           sales.saleprice,                                                     --Tatsächlichen Verkaufspreis auswählen
           sales.saledate,                                                      --Verkaufsdatum auswählen
           customers.firstname AS customer_firstname,                           --Kundenvornamen eindeutig benennen
           customers.lastname AS customer_lastname,                             --Kundennachnamen eindeutig benennen
           salespersons.firstname AS salesperson_firstname,                     --Verkäufervornamen eindeutig benennen
           salespersons.lastname AS salesperson_lastname                        --Verkäufernachnamen eindeutig benennen
    FROM bicycles                                                               --Von der Fahrradtabelle ausgehen
    INNER JOIN sales                                                            --Verkaufsinformationen ergänzen
        ON bicycles.bicycleid = sales.bicycleid                                 --Fahrrad und Verkauf über die Fahrrad-ID verbinden
    INNER JOIN customers                                                        --Kundeninformationen ergänzen
        ON sales.customerid = customers.customerid                              --Verkauf und Kunde über die Kunden-ID verbinden
    INNER JOIN salespersons                                                     --Verkäuferinformationen ergänzen
        ON sales.salespersonid = salespersons.salespersonid                     --Verkauf und Verkäufer über die Verkäufer-ID verbinden
    WHERE bicycles.make = 'Bianchi'                                             --Nur Fahrräder der Marke Bianchi berücksichtigen
      AND sales.saledate >= '2023-03-01'                                        --Verkäufe ab dem 1. März 2023 einschliessen
      AND sales.saledate < '2024-01-01'                                         --Bis Ende Dezember einschliessen, auch bei gespeicherten Uhrzeiten
    ORDER BY sales.saledate ASC                                                 --Älteste Verkäufe zuerst anzeigen
    LIMIT 10;                                                                   --Ausgabe auf maximal zehn Zeilen begrenzen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,saleprice,saledate,customer_firstname,customer_lastname,salesperson_firstname,salesperson_lastname


### Determine the number of services, average service cost and most recent service date for each mechanic and bicycle make across all years. Display the mechanic's first and last name, make, services_count, average_service_costs and last_service_date. Round average costs to two decimal places and sort them in descending order
- Zwei INNER JOINs verbinden Mechaniker, Services und Fahrräder, gruppiert wird pro Mechaniker und Marke über alle Jahre
- `COUNT` zählt Services, `AVG(cost)` berechnet den gerundeten Kostendurchschnitt und `MAX(servicedate)` das jüngste Servicedatum, sortiert wird nach Durchschnittskosten absteigend
- Format: `SELECT Tabelle1.*, Tabelle2.* FROM Tabelle1 INNER JOIN Tabelle2 ON Tabelle1.GemeinsameSpalte = Tabelle2.GemeinsameSpalte;`, für die dritte Tabelle einen weiteren INNER JOIN ergänzen

In [28]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT mechanics.firstname AS mechanic_firstname,                                       --Vornamen des Mechanikers anzeigen
           mechanics.lastname AS mechanic_lastname,                                         --Nachnamen des Mechanikers anzeigen
           bicycles.make,                                                                   --Fahrradmarke anzeigen
           COUNT(*) AS services_count,                                                      --Services pro Mechaniker und Marke zählen
           ROUND(CAST(AVG(servicerecords.cost) AS NUMERIC), 2) AS average_service_costs,    --Durchschnittliche Servicekosten auf zwei Stellen runden
           MAX(servicerecords.servicedate) AS last_service_date                             --Neuestes Servicedatum innerhalb der Gruppe ermitteln
    FROM mechanics                                                                          --Von der Mechanikertabelle ausgehen
    INNER JOIN servicerecords                                                               --Durchgeführte Services ergänzen
        ON mechanics.mechanicid = servicerecords.mechanicid                                 --Services über die Mechaniker-ID zuordnen
    INNER JOIN bicycles                                                                     --Informationen zu den gewarteten Fahrrädern ergänzen
        ON servicerecords.bicycleid = bicycles.bicycleid                                    --Services über die Fahrrad-ID zuordnen
    GROUP BY mechanics.mechanicid, mechanics.firstname, mechanics.lastname, bicycles.make   --Nach Mechaniker und Marke gruppieren, keine Gruppierung nach Jahr
    ORDER BY average_service_costs DESC;                                                    --Höchste durchschnittliche Servicekosten zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,mechanic_firstname,mechanic_lastname,make,services_count,average_service_costs,last_service_date
0,Shannon,Larsen,Scott,65,39.76,2024-08-07
1,Cynthia,Pittman,Cannondale,74,38.61,2024-07-28
2,Shannon,Larsen,Specialized,75,37.41,2024-08-20
3,Courtney,Ford,Santa Cruz,82,36.41,2024-08-04
4,Erin,Williamson,Specialized,69,36.14,2024-08-17
...,...,...,...,...,...,...
95,Cynthia,Pittman,Specialized,82,27.53,2024-08-19
96,Lisa,Hall,Bianchi,70,27.47,2024-08-26
97,Cynthia,Pittman,Bianchi,77,26.81,2024-07-20
98,Melissa,Miller,Bianchi,71,26.49,2024-08-26


### Determine the average cost per year for services described as "Gear shifting adjustment". Display year and average_service_costs, rounded to two decimal places and sorted by average_service_costs in descending order
- `WHERE` filtert auf den gewünschten Service, `CAST(EXTRACT(YEAR FROM servicedate) AS INTEGER)` liest das Jahr als Ganzzahl aus und dient zur Gruppierung
- `AVG(cost)` berechnet die Durchschnittskosten pro Jahr, die Ausgabe wird auf zwei Dezimalstellen gerundet und nach Durchschnittskosten absteigend sortiert
- Kein JOIN nötig, alle benötigten Spalten stehen in `servicerecords`

In [29]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT CAST(EXTRACT(YEAR FROM servicedate) AS INTEGER) AS year,             --Jahr aus dem Servicedatum auslesen und als Ganzzahl ausgeben
           ROUND(CAST(AVG(cost) AS NUMERIC), 2) AS average_service_costs        --Durchschnittliche Servicekosten pro Jahr auf zwei Stellen runden
    FROM servicerecords                                                         --Daten aus der Tabelle servicerecords abrufen
    WHERE servicedescription = 'Gear shifting adjustment'                       --Nur Services mit genau dieser Beschreibung berücksichtigen
    GROUP BY CAST(EXTRACT(YEAR FROM servicedate) AS INTEGER)                    --Alle passenden Services desselben Jahres zusammenfassen
    ORDER BY average_service_costs DESC;                                        --Jahr mit den höchsten durchschnittlichen Kosten zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,year,average_service_costs
0,2019,25.83
1,2021,25.71
2,2020,25.40
3,2022,25.30
4,2023,25.07
5,2024,24.59


### Determine the number of tickets sold for every cinema, including cinemas without screenings or tickets. Display cinemaid, name and ticket_count, sorted by ticket_count in descending order
- Zwei LEFT JOINs verbinden `cinemas → screenings → tickets`, alle Kinos bleiben erhalten
- `COUNT(t.ticketid)` zählt nur echte Tickets und liefert ohne Tickets 0, `COUNT(*)` würde ergänzte NULL-Zeilen mitzählen
- Format: `SELECT Tabelle1.*, Tabelle2.* FROM Tabelle1 LEFT JOIN Tabelle2 ON Tabelle1.GemeinsameSpalte = Tabelle2.GemeinsameSpalte;`, für die dritte Tabelle einen weiteren LEFT JOIN ergänzen

In [30]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT cinemas.cinemaid,                                                    --Kino-ID auswählen
           cinemas.name,                                                        --Kinonamen auswählen
           COUNT(tickets.ticketid) AS ticket_count                              --Nur vorhandene Tickets zählen und ohne Treffer 0 ausgeben
    FROM cinemas                                                                --Mit allen Kinos als Ausgangstabelle beginnen
    LEFT JOIN screenings                                                        --Vorstellungen ergänzen und Kinos ohne Vorstellung behalten
        ON cinemas.cinemaid = screenings.cinemaid                               --Kinos und Vorstellungen über die Kino-ID verbinden
    LEFT JOIN tickets                                                           --Tickets ergänzen und Zeilen ohne Ticket behalten
        ON screenings.screeningid = tickets.screeningid                         --Vorstellungen und Tickets über die Vorstellungs-ID verbinden
    GROUP BY cinemas.cinemaid, cinemas.name                                     --Eine Gruppe pro Kino bilden
    ORDER BY ticket_count DESC, cinemas.cinemaid ASC;                           --Meiste Tickets zuerst und bei Gleichstand nach Kino-ID sortieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,cinemaid,name,ticket_count
0,C005,Kinok,441
1,C002,Pathé Küchlin,436
2,C004,Rex Kino,386
3,C003,Cineplexx,383
4,C001,Arthouse Metropol,354
5,C006,Kino Aurora,0


### Identify cinemas that have sold more than 400 tickets. Display name and ticket_count, sorted by ticket_count in descending order
- Zwei INNER JOINs verbinden `cinemas → screenings → tickets`, `GROUP BY` bildet eine Gruppe pro Kino
- `HAVING COUNT(t.ticketid) > 400` behält nur Kinos mit mehr als 400 Tickets, genau 400 ist ausgeschlossen
- Format: `SELECT Tabelle1.*, Tabelle2.* FROM Tabelle1 INNER JOIN Tabelle2 ON Tabelle1.GemeinsameSpalte = Tabelle2.GemeinsameSpalte;`, für die dritte Tabelle einen weiteren INNER JOIN ergänzen

In [31]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT cinemas.name,                                                        --Kinonamen auswählen
           COUNT(tickets.ticketid) AS ticket_count                              --Verkaufte Tickets je Kino zählen
    FROM cinemas                                                                --Kinos als Ausgangstabelle verwenden
    INNER JOIN screenings                                                       --Passende Vorstellungen ergänzen
        ON cinemas.cinemaid = screenings.cinemaid                               --Kinos und Vorstellungen über die Kino-ID verbinden
    INNER JOIN tickets                                                          --Passende Tickets ergänzen
        ON screenings.screeningid = tickets.screeningid                         --Vorstellungen und Tickets über die Vorstellungs-ID verbinden
    GROUP BY cinemas.cinemaid, cinemas.name                                     --Nach Kino-ID und Name gruppieren damit gleichnamige Kinos getrennt bleiben
    HAVING COUNT(tickets.ticketid) > 400                                        --Nur Kinos mit mehr als 400 verkauften Tickets behalten
    ORDER BY ticket_count DESC, cinemas.cinemaid ASC;                           --Meiste Tickets zuerst und bei Gleichstand nach Kino-ID sortieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,name,ticket_count
0,Kinok,441
1,Pathé Küchlin,436
